# NER eğitimi ve değerlendirme (kişi / kurum / yer / adres)

**Hazırlık (bir kez):** `turkce-dlp-colab-bundle-v2` paketini (zip ya da Safari'nin açtığı klasör)
Google Drive'da `MyDrive/turkce-llm-dlp/` klasörüne yükle.

**Çalıştırma:** Çalışma zamanı → türü **T4 GPU** → **Tümünü çalıştır**.
Aşağıdaki ayar hücresinde hangi deneyin çalışacağını seç. Drive'da aynı adlı model
zaten varsa eğitim atlanır ve yalnızca değerlendirme yapılır.

In [ ]:
# 0) Deney ayarı — yalnızca burayı değiştir
DENEY = 'augmented'   # 'augmented' (zenginleştirilmiş veri) veya 'baseline' (eski veri)
SEED = 42

DATA_DIR = {'augmented': 'data/ner/processed/augmented-v1',
            'baseline': 'data/ner/processed/real-address-context-v1'}[DENEY]
RUN = f'{DENEY}_seed{SEED}' if DENEY == 'augmented' else f'baseline_ctx_seed{SEED}'
print('Deney:', RUN, '| veri:', DATA_DIR)

In [ ]:
# 1) GPU ve Drive
import torch
assert torch.cuda.is_available(), "GPU yok: Çalışma zamanı türünü T4 GPU yap ve yeniden başlat."
print(torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Paketi Drive'dan al (klasör veya zip; en yeni sürüm tercih edilir)
import zipfile, pathlib, shutil
DRIVE = pathlib.Path('/content/drive/MyDrive/turkce-llm-dlp')
DRIVE.mkdir(parents=True, exist_ok=True)
shutil.rmtree('/content/proje', ignore_errors=True)
candidates = [DRIVE / 'turkce-dlp-colab-bundle-v2', DRIVE / 'turkce-dlp-colab-bundle']
zips = sorted(pathlib.Path('/content/drive/MyDrive').rglob('turkce-dlp-colab-bundle*.zip'), reverse=True)
folder = next((c for c in candidates if c.is_dir()), None)
if folder:
    shutil.copytree(folder, '/content/proje')
elif zips:
    with zipfile.ZipFile(zips[0]) as zf:
        zf.extractall('/content/proje')
else:
    raise FileNotFoundError("Paket bulunamadı; Drive'da MyDrive/turkce-llm-dlp/ klasörüne yükle.")
%cd /content/proje
!cat BUNDLE_INFO.txt
assert pathlib.Path(DATA_DIR).exists(), f'{DATA_DIR} pakette yok — v2 paketini yükledin mi?'

In [ ]:
# 3) Bağımlılıklar ve veri doğrulama
!pip -q install -r training/ner/requirements.txt
!USE_TF=0 python training/ner/train.py --data-dir {DATA_DIR} --labels-path training/ner/labels.json --validate-only > /dev/null && echo 'Veri geçerli'

In [ ]:
# 4) Eğitim (Drive'da model varsa atlanır). T4'te yaklaşık 45-75 dk.
OUT = f'/content/runs/{RUN}'
saved = DRIVE / 'runs' / RUN
if (saved / 'model').exists():
    print('Drive\'da hazır model bulundu, eğitim atlanıyor:', saved)
    shutil.rmtree(OUT, ignore_errors=True)
    shutil.copytree(saved, OUT)
else:
    !USE_TF=0 python training/ner/train.py --data-dir {DATA_DIR} --labels-path training/ner/labels.json --output-dir {OUT} --epochs 3 --seed {SEED} --fp16
    shutil.rmtree(saved, ignore_errors=True)
    shutil.copytree(OUT, saved, ignore=shutil.ignore_patterns('checkpoints'))
    print('Drive\'a kaydedildi:', saved)

In [ ]:
# 5) Ek testler: cümle içi OSM adresleri + ALTIN TEST SETİ (gerçekçi, elle doğrulanmış)
!USE_TF=0 python training/ner/evaluate_model.py --model-dir {OUT}/model --data-dir data/ner/processed/openstreetmap-tr/sentence-context-v1 --split test --output-json {OUT}/context-test-metrics.json > /dev/null
!USE_TF=0 python training/ner/evaluate_model.py --model-dir {OUT}/model --data-dir data/ner/processed/realistic-v1 --split test --output-json {OUT}/gold-test-metrics.json > /dev/null
for name in ('context-test-metrics.json', 'gold-test-metrics.json'):
    shutil.copy(f'{OUT}/{name}', saved / name)
print('Değerlendirme tamam')

In [ ]:
# 6) Bana yapıştırılacak özet (yalnızca sayılar, ham metin yok)
import json
def pick(obj, path=''):
    out = {}
    if isinstance(obj, dict):
        for k, v in obj.items():
            out.update(pick(v, f'{path}.{k}' if path else k))
    elif isinstance(obj, (int, float)) and any(x in path for x in ('precision', 'recall', 'f1')):
        out[path] = round(obj, 4)
    return out
summary = {'run': RUN}
summary['train_metrics'] = {k: v for k, v in pick(json.load(open(f'{OUT}/metrics.json'))).items() if 'test' in k}
for name in ('context', 'gold'):
    data = json.load(open(f'{OUT}/{name}-test-metrics.json'))
    summary[f'{name}_test'] = pick(data.get('metrics', data))
print(json.dumps(summary, ensure_ascii=False, indent=1))